In [1]:
import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface"), ignore_errors=True)
print("완료")

완료


In [2]:
!pip install monai==1.3.1 transformers==4.40.0 numpy==1.26.4 nibabel==5.2.1 SimpleITK==2.3.1 einops==0.8.0 peft==0.8.2 safetensors==0.4.3 accelerate

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "GoodBaiBai88/M3D-LaMed-Phi-3-4B"
dtype = torch.bfloat16

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    model_max_length=512,
    padding_side="right",
    use_fast=False,
    trust_remote_code=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map='auto',
    trust_remote_code=True
)
print("로드 성공")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


modeling_m3d_lamed.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/GoodBaiBai88/M3D-LaMed-Phi-3-4B:
- modeling_m3d_lamed.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.blocks.patchembedding PatchEmbeddingBlock.__init__:pos_embed: Argument `pos_embed` has been deprecated since version 1.2. It will be removed in version 1.4. please use `proj_type` instead.
  warn_deprecated(argname, msg, warning_category)
/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:221: FutureWarning: monai.networks.nets.vit ViT.__init__:pos_embed: Argument `pos_embed` has been deprecated since version 1.2. It will be removed in version 1.4. please use `proj_type` instead.
  warn_deprecated(argname, msg, warning_category)


build_sam_vit_3d...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]

로드 성공


In [2]:
!pip install transformers==4.41.0
import shutil, os
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/modules"), ignore_errors=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 92.6 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.40.0
    Uninstalling transformers-4.40.0:
      Successfully uninstalled transformers-4.40.0


In [2]:
!git clone https://github.com/BAAI-DCAI/M3D.git

fatal: destination path 'M3D' already exists and is not an empty directory.


In [3]:
import numpy as np
import torch

device = torch.device('cuda')
proj_out_num = 256

image_path = "M3D/Data/data/examples/example_03.npy"
question = "What is liver in this image? Please output the segmentation mask."

image_tokens = "<im_patch>" * proj_out_num
input_txt = image_tokens + question
input_id = tokenizer(input_txt, return_tensors="pt")['input_ids'].to(device=device)

image_np = np.load(image_path)
image_pt = torch.from_numpy(image_np).unsqueeze(0).to(dtype=torch.bfloat16, device=device)

generation, seg_logit = model.generate(image_pt, input_id, seg_enable=True, max_new_tokens=256, do_sample=True, top_p=0.9, temperature=1.0)

generated_texts = tokenizer.batch_decode(generation, skip_special_tokens=True)
seg_mask = (torch.sigmoid(seg_logit) > 0.5) * 1.0

print('question:', question)
print('generated_texts:', generated_texts[0])
print('seg_mask shape:', seg_mask.shape)

You are not running the flash-attention implementation, expect numerical differences.


question: What is liver in this image? Please output the segmentation mask.
generated_texts: Sure,  [SEG] .
seg_mask shape: torch.Size([1, 1, 32, 256, 256])


In [4]:
import SimpleITK as sitk
import numpy as np

mask_np = seg_mask.cpu().numpy()[0][0]  # (32, 256, 256)

voxel_count = np.sum(mask_np)
spacing = (1.0, 1.0, 1.0)  # 샘플이라 spacing 모르니 일단 1mm로

volume_cm3 = voxel_count * spacing[0] * spacing[1] * spacing[2] / 1000
print(f"Voxel count: {int(voxel_count)}")
print(f"Volume: {volume_cm3:.2f} cm³")

Voxel count: 62214
Volume: 62.21 cm³
